# FIS Finance IQ - synthetic FP&A demo

As of 2026-09-30. Not FIS financial results. This notebook overwrites only the dedicated demo Lakehouses. No schedules, notifications or source-system writes.

In [ ]:
WORKSPACE_ID = 'f00e31f4-f4ab-435c-87f4-fdb5f4b06c4a'
BRONZE_ID = 'ad139c93-78be-4c31-a249-375f1ba4c65d'
SILVER_ID = 'e15b7040-5c9f-46a0-9ada-d4bc69ba6ad3'
GOLD_ID = '3773425f-3a99-4bda-be3f-acecfcabb0df'


In [ ]:
import json
from datetime import date, datetime
from decimal import Decimal
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DecimalType

namespace = {"__name__": "fis_finance_data", "__file__": "/lakehouse/default/Files/fis_finance_iq/data.py"}
exec(compile('"""Deterministic, fictional FIS-style FP&A data. No customer data is used."""\n\nfrom __future__ import annotations\n\nimport calendar\nimport csv\nimport hashlib\nimport json\nfrom datetime import date, timedelta\nfrom decimal import Decimal\nfrom pathlib import Path\n\n\nROOT = Path(__file__).resolve().parent\nAS_OF = date(2026, 9, 30)\nYEAR = AS_OF.year\nFACTS = ("ACTUALS", "FORECAST", "PLAN")\nKEYS = {\n    "account": "Account",\n    "cost_center": "CostCenter",\n    "DIM_DATE": "DATE",\n    "product": "ProductLine",\n    **{name: "RecordID" for name in FACTS},\n}\nDIMENSIONS = (\n    ("account", "ACCOUNT", "Account"),\n    ("cost_center", "COSTCENTER", "CostCenter"),\n    ("DIM_DATE", "DATE", "DATE"),\n    ("product", "PRODUCTLINE", "ProductLine"),\n)\nRELATIONSHIPS = [\n    {"fromTable": fact, "fromColumn": column, "toTable": dimension, "toColumn": key}\n    for fact in FACTS\n    for dimension, column, key in DIMENSIONS\n]\nScalar = str | int\nRow = dict[str, Scalar]\nTables = dict[str, list[Row]]\n\n\ndef month_date(year: int, month: int) -> str:\n    return date(year, month, 1).isoformat()\n\n\ndef baseline(account: Row, product: Row, center: Row, month: int) -> int:\n    return (\n        int(account["MonthlyBaseCents"])\n        * int(product["DemoWeight"])\n        * int(center["DemoWeight"])\n        * (100 + 2 * month)\n        // 1_000_000\n    )\n\n\ndef scenario_amount(\n    planned: int, account: Row, product: Row, center: Row, month: int, future: bool = False\n) -> int:\n    if account["AccountType"] == "Revenue":\n        factor = 9700\n        if product["ProductLine"] == "DIGITAL" and month >= 7:\n            factor = 9300 if future else 9000\n    elif account["Account"] == "500100":\n        factor = 10300\n        if center["CostCenter"] == "300110" and product["ProductLine"] in {"DIGITAL", "PAYMENTS"}:\n            factor = (13000 if future else 14500) if month >= 6 else 10500\n    elif account["Account"] == "500200":\n        factor = 9800 if future else 9600\n    else:\n        factor = 10200\n    return planned * factor // 10000\n\n\ndef fact_row(\n    scenario: str, year: int, month: int, account: Row, product: Row, center: Row, amount: int\n) -> Row:\n    account_id, product_id, center_id = account["Account"], product["ProductLine"], center["CostCenter"]\n    versions = {"ACTUALS": "ACTUAL_FINAL_2026_09", "FORECAST": "FCST_2026_09", "PLAN": "PLAN_2026_V1"}\n    return {\n        "RecordID": f"{scenario}-{year}{month:02}-{account_id}-{center_id}-{product_id}",\n        "ACCOUNT": account_id,\n        "Amount": str(Decimal(amount).scaleb(-2)),\n        "AmountCents": amount,\n        "CC_PR_AC": f"{center_id}|{product_id}|{account_id}",\n        "COSTCENTER": center_id,\n        "COSTCENTER_PRODUCTLINE": f"{center_id}|{product_id}",\n        "CURRENCY": "USD",\n        "CUSTOMER": "SYNTHETIC_AGGREGATE",\n        "DATASOURCE": "SYNTHETIC_FIS_DEMO",\n        "DATE": month_date(year, month),\n        "ENTITY": "FIS_DEMO_US",\n        "ICP": "EXTERNAL",\n        "LOAD_DATE": AS_OF.isoformat(),\n        "LOAD_TIMESTAMP": f"{AS_OF.isoformat()}T23:00:00Z",\n        "LOCATION": "US",\n        "PERIOD": month,\n        "PRODUCTLINE": product_id,\n        "QUARTER": f"Q{(month - 1) // 3 + 1}",\n        "SCENARIO": scenario,\n        "VERSION": versions[scenario],\n        "VIEW": "PERIODIC",\n        "YEAR": year,\n    }\n\n\ndef build_data() -> Tables:\n    accounts: list[Row] = []\n    account_specs = [\n        ("400100", "Platform subscriptions", "Revenue", 120_000_000),\n        ("400200", "Transaction processing", "Revenue", 90_000_000),\n        ("400300", "Professional services", "Revenue", 25_000_000),\n        ("500100", "Cloud and infrastructure", "Operating Expense", 26_000_000),\n        ("500200", "Workforce", "Operating Expense", 65_000_000),\n        ("500300", "Third-party services", "Operating Expense", 19_000_000),\n        ("500400", "Facilities", "Operating Expense", 6_000_000),\n        ("500500", "Risk and compliance", "Operating Expense", 9_000_000),\n    ]\n    for code, description, category, base in account_specs:\n        row: Row = {\n            "Account": code, "Description": description, "Description_raw": description,\n            "AccountType": category, "FavorableSign": 1 if category == "Revenue" else -1,\n            "MonthlyBaseCents": base,\n        }\n        for level, value in enumerate(\n            ["FIS demo", "Operating statement", category, category, description, description, description, code], 1\n        ):\n            row[f"L{level}"] = value\n            row[f"L{level}_Description"] = value\n        accounts.append(row)\n\n    products: list[Row] = []\n    for code, description, segment, weight in [\n        ("BNK_CORE", "Core Banking Platform", "Banking", 125),\n        ("DIGITAL", "Digital Banking", "Banking", 110),\n        ("PAYMENTS", "Payment Processing", "Banking", 145),\n        ("CAP_MKT", "Capital Markets Platform", "Capital Markets", 130),\n        ("LENDING", "Lending Services", "Banking", 90),\n        ("TREASURY", "Treasury Solutions", "Capital Markets", 85),\n    ]:\n        row = {\n            "ProductLine": code, "ProductLine_Cleaned": code, "Description": description,\n            "Description_New": description, "Segment": segment, "DemoWeight": weight,\n        }\n        for level, value in enumerate(\n            ["FIS demo", segment, "Financial technology", description, "Solutions", "Global", description, code], 1\n        ):\n            row[f"L{level}"] = value\n            row[f"L{level}_Description"] = value\n            row[f"L{level}_Description_New"] = value\n        products.append(row)\n\n    centers: list[Row] = []\n    for code, description, function, weight in [\n        ("100110", "Banking Engineering", "Engineering", 100),\n        ("100210", "Digital Operations", "Operations", 85),\n        ("200110", "Markets Operations", "Operations", 110),\n        ("300110", "Cloud Infrastructure", "Technology", 140),\n        ("900110", "Corporate Finance", "Corporate", 55),\n        ("900210", "Risk and Compliance", "Corporate", 60),\n    ]:\n        row = {\n            "CostCenter": code, "CostCenter_Cleaned": code,\n            "Description": description, "Function": function, "DemoWeight": weight,\n        }\n        for level, value in enumerate(["FIS demo", function, description, code], 1):\n            row[f"CC_L{level}"] = value\n            row[f"CC_L{level}_Description"] = value\n            row[f"CC_L{level}_Description_New"] = value\n        centers.append(row)\n\n    dates: list[Row] = []\n    current = date(YEAR - 1, 1, 1)\n    while current <= date(YEAR, 12, 31):\n        quarter = (current.month - 1) // 3 + 1\n        first = current.replace(day=1)\n        last = current.replace(day=calendar.monthrange(current.year, current.month)[1])\n        first_quarter = date(current.year, (quarter - 1) * 3 + 1, 1)\n        last_quarter = date(current.year, quarter * 3, calendar.monthrange(current.year, quarter * 3)[1])\n        prior = current.replace(year=current.year - 1)\n        dates.append({\n            "DATE": current.isoformat(), "DATE_INTEGER": int(current.strftime("%Y%m%d")),\n            "DAY_NUMBER": current.day, "DAY_OF_WEEK": current.isoweekday(),\n            "DAY_OF_WEEK_LONG": current.strftime("%A"), "DAY_OF_WEEK_SHORT": current.strftime("%a"),\n            "FIRST_DAY_MONTH": first.isoformat(), "FIRST_DAY_QUARTER": first_quarter.isoformat(),\n            "LAST_DAY_MONTH": last.isoformat(), "LAST_DAY_QUARTER": last_quarter.isoformat(),\n            "MONTH_ID": current.year * 100 + current.month,\n            "MONTH_NAME_LONG": current.strftime("%B"), "MONTH_NAME_SHORT": current.strftime("%b"),\n            "MONTH_YEAR": current.strftime("%b %Y"), "MONTH_YEAR_INTEGER": current.year * 100 + current.month,\n            "MONTHNUMBER": current.month, "PY_COMPARE_DATE": prior.isoformat(),\n            "QUARTER": f"Q{quarter}", "QUARTER_ID": current.year * 10 + quarter,\n            "QUARTER_SEQUENCE": (current.year - (YEAR - 1)) * 4 + quarter,\n            "REPORTING_MONTH": AS_OF.strftime("%Y-%m"),\n            "SOW_MONDAY": (current - timedelta(days=current.weekday())).isoformat(),\n            "WEEK_OF_YEAR": current.isocalendar().week,\n            "YEAR": current.year, "YEAR_MONTH": current.strftime("%Y-%m"),\n            "YEAR_MONTH_SHORT": current.strftime("%b %y"), "YEAR_MONTH_SHORT_V1": current.strftime("%b-%y"),\n            "YEAR_QUARTER": f"{current.year} Q{quarter}",\n        })\n        current += timedelta(days=1)\n\n    tables: Tables = {"account": accounts, "cost_center": centers, "DIM_DATE": dates, "product": products}\n    tables.update({name: [] for name in FACTS})\n    for account in accounts:\n        for product in products:\n            for center in centers:\n                for month in range(1, 13):\n                    planned = baseline(account, product, center, month)\n                    prior = planned * (9000 + month * 10) // 10000\n                    tables["ACTUALS"].append(fact_row("ACTUALS", YEAR - 1, month, account, product, center, prior))\n                    tables["PLAN"].append(fact_row("PLAN", YEAR, month, account, product, center, planned))\n                    actual = scenario_amount(planned, account, product, center, month)\n                    if month <= AS_OF.month:\n                        tables["ACTUALS"].append(fact_row("ACTUALS", YEAR, month, account, product, center, actual))\n                    outlook = scenario_amount(planned, account, product, center, month, future=month > AS_OF.month)\n                    tables["FORECAST"].append(fact_row("FORECAST", YEAR, month, account, product, center, outlook))\n    return tables\n\n\ndef column_type(name: str, value: Scalar) -> str:\n    if name == "Amount":\n        return "decimal"\n    if name == "LOAD_TIMESTAMP":\n        return "timestamp"\n    if name in {\n        "DATE", "LOAD_DATE", "FIRST_DAY_MONTH", "FIRST_DAY_QUARTER",\n        "LAST_DAY_MONTH", "LAST_DAY_QUARTER", "PY_COMPARE_DATE", "SOW_MONDAY",\n    }:\n        return "date"\n    return "int64" if isinstance(value, int) else "string"\n\n\ndef totals(tables: Tables, fact: str, category: str, ytd: bool = False, **filters: Scalar) -> int:\n    accounts = {row["Account"] for row in tables["account"] if row["AccountType"] == category}\n    return sum(\n        int(row["AmountCents"]) for row in tables[fact]\n        if row["ACCOUNT"] in accounts and row["YEAR"] == YEAR\n        and (not ytd or int(row["PERIOD"]) <= AS_OF.month)\n        and all(row[key] == value for key, value in filters.items())\n    )\n\n\ndef metrics(tables: Tables) -> dict[str, int | float]:\n    result: dict[str, int | float] = {}\n    for fact, prefix in (("ACTUALS", "Actual"), ("PLAN", "Plan"), ("FORECAST", "Forecast")):\n        for ytd in ((True,) if fact == "ACTUALS" else (False, True)):\n            suffix = "YTD" if ytd else "FY"\n            revenue = totals(tables, fact, "Revenue", ytd)\n            expense = totals(tables, fact, "Operating Expense", ytd)\n            result[f"{prefix}Revenue{suffix}"] = revenue / 100\n            result[f"{prefix}Opex{suffix}"] = expense / 100\n            result[f"{prefix}Profit{suffix}"] = (revenue - expense) / 100\n    result["RevenueVarianceYTD"] = round(result["ActualRevenueYTD"] - result["PlanRevenueYTD"], 2)\n    result["ExpenseVarianceYTD"] = round(result["PlanOpexYTD"] - result["ActualOpexYTD"], 2)\n    result["ProfitVarianceYTD"] = round(result["ActualProfitYTD"] - result["PlanProfitYTD"], 2)\n    result["ForecastProfitVarianceFY"] = round(result["ForecastProfitFY"] - result["PlanProfitFY"], 2)\n    hero = {"ACCOUNT": "500100", "PRODUCTLINE": "DIGITAL", "COSTCENTER": "300110"}\n    result["CloudDigitalActualYTD"] = totals(tables, "ACTUALS", "Operating Expense", True, **hero) / 100\n    result["CloudDigitalPlanYTD"] = totals(tables, "PLAN", "Operating Expense", True, **hero) / 100\n    return result\n\n\ndef validate(tables: Tables) -> dict:\n    if set(tables) != set(KEYS):\n        raise ValueError(f"Expected exactly seven finance tables, received {sorted(tables)}")\n    for name, key in KEYS.items():\n        values = [row[key] for row in tables[name]]\n        if not values or any(value == "" for value in values) or len(set(values)) != len(values):\n            raise ValueError(f"{name}.{key} must be populated and unique")\n    checks = 0\n    for relationship in RELATIONSHIPS:\n        keys = {row[relationship["toColumn"]] for row in tables[relationship["toTable"]]}\n        rows = tables[relationship["fromTable"]]\n        if any(row[relationship["fromColumn"]] not in keys for row in rows):\n            raise ValueError(f"Orphan foreign key: {relationship}")\n        checks += len(rows)\n    grains: dict[str, dict[tuple, Row]] = {}\n    for fact in FACTS:\n        grains[fact] = {}\n        for row in tables[fact]:\n            if row["SCENARIO"] != fact or row["CURRENCY"] != "USD":\n                raise ValueError(f"Invalid scenario or currency in {row[\'RecordID\']}")\n            if Decimal(str(row["Amount"])) * 100 != row["AmountCents"]:\n                raise ValueError(f"Amount and AmountCents disagree in {row[\'RecordID\']}")\n            if fact == "ACTUALS" and date.fromisoformat(str(row["DATE"])) > AS_OF:\n                raise ValueError("ACTUALS contains an unclosed period")\n            grain = tuple(row[key] for key in ("ACCOUNT", "COSTCENTER", "PRODUCTLINE", "DATE"))\n            if grain in grains[fact]:\n                raise ValueError(f"Duplicate finance grain in {fact}: {grain}")\n            grains[fact][grain] = row\n        expected_months = (\n            {month_date(YEAR - 1, month) for month in range(1, 13)}\n            | {month_date(YEAR, month) for month in range(1, AS_OF.month + 1)}\n        ) if fact == "ACTUALS" else {month_date(YEAR, month) for month in range(1, 13)}\n        for account in tables["account"]:\n            for center in tables["cost_center"]:\n                for product in tables["product"]:\n                    actual_months = {\n                        grain[3] for grain in grains[fact]\n                        if grain[:3] == (account["Account"], center["CostCenter"], product["ProductLine"])\n                    }\n                    if actual_months != expected_months:\n                        raise ValueError(f"Incomplete period coverage in {fact}")\n    for grain, row in grains["ACTUALS"].items():\n        if row["YEAR"] == YEAR and grains["FORECAST"][grain]["AmountCents"] != row["AmountCents"]:\n            raise ValueError(f"Closed-month forecast does not match ACTUALS: {grain}")\n    values = metrics(tables)\n    if round(values["RevenueVarianceYTD"] + values["ExpenseVarianceYTD"], 2) != round(values["ProfitVarianceYTD"], 2):\n        raise ValueError("Revenue and favorable expense variances do not reconcile to profit variance")\n    return {\n        "status": "PASS", "synthetic": True, "asOf": AS_OF.isoformat(),\n        "tableRows": {name: len(rows) for name, rows in tables.items()},\n        "relationships": len(RELATIONSHIPS), "foreignKeyChecks": checks, "metrics": values,\n    }\n\n\ndef write_data(target: Path = ROOT / "generated") -> dict:\n    tables = build_data()\n    report = validate(tables)\n    csv_dir = target / "data"\n    csv_dir.mkdir(parents=True, exist_ok=True)\n    hashes = {}\n    schema = {}\n    for name, rows in tables.items():\n        path = csv_dir / f"{name}.csv"\n        with path.open("w", newline="", encoding="utf-8") as stream:\n            writer = csv.DictWriter(stream, fieldnames=list(rows[0]), lineterminator="\\n")\n            writer.writeheader()\n            writer.writerows(rows)\n        hashes[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()\n        schema[name] = {column: column_type(column, value) for column, value in rows[0].items()}\n    for name, value in (\n        ("validation.json", report), ("schema.json", schema),\n        ("relationships.json", RELATIONSHIPS), ("data-sha256.json", hashes),\n    ):\n        (target / name).write_text(json.dumps(value, indent=2) + "\\n", encoding="utf-8")\n    return report\n\n\nif __name__ == "__main__":\n    print(json.dumps(write_data(), indent=2))\n', namespace["__file__"], "exec"), namespace)
data = namespace["build_data"]()
expected = namespace["validate"](data)
base = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com"
roots = {"bronze": f"{base}/{BRONZE_ID}", "silver": f"{base}/{SILVER_ID}", "gold": f"{base}/{GOLD_ID}"}
notebookutils.fs.mkdirs(roots["gold"] + "/Files/fis_finance_iq")
print(json.dumps(expected, indent=2))


## Bronze: deterministic source ledger

In [ ]:
for name, rows in data.items():
    fields = []
    typed_rows = [dict(row) for row in rows]
    for column, value in rows[0].items():
        kind = namespace["column_type"](column, value)
        datatype = LongType() if kind == "int64" else DecimalType(18,2) if kind == "decimal" else StringType()
        fields.append(StructField(column, datatype, False))
        if kind == "decimal":
            for row in typed_rows:
                row[column] = Decimal(str(row[column]))
    frame = spark.createDataFrame(typed_rows, StructType(fields))
    frame.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(roots["bronze"] + "/Tables/" + name.lower())
    print("BRONZE", name, frame.count())


## Silver: typed, validated finance data

In [ ]:
for name, rows in data.items():
    frame = spark.read.format("delta").load(roots["bronze"] + "/Tables/" + name.lower())
    for column, value in rows[0].items():
        kind = namespace["column_type"](column, value)
        if kind == "date":
            frame = frame.withColumn(column, F.to_date(column))
        elif kind == "timestamp":
            frame = frame.withColumn(column, F.to_timestamp(column))
        elif kind == "string":
            frame = frame.withColumn(column, F.trim(F.col(column)))
    null_counts = frame.select([
        F.sum(F.col(column).isNull().cast("int")).alias(column) for column in frame.columns
    ]).first().asDict()
    invalid = [column for column, count in null_counts.items() if count]
    if invalid:
        raise ValueError(f"Invalid or missing values in {name}: {invalid}")
    key = namespace["KEYS"][name]
    if frame.select(key).distinct().count() != len(rows):
        raise ValueError(f"Duplicate key in {name}")
    frame.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(roots["silver"] + "/Tables/" + name.lower())
    print("SILVER", name, frame.count())


## Gold: conformed star schema, variance drivers, relationship edges

In [ ]:
spark.conf.set("spark.sql.parquet.vorder.default", "true")
for name in data:
    frame = spark.read.format("delta").load(roots["silver"] + "/Tables/" + name.lower())
    frame.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(name.lower())
    print("GOLD", name, spark.table(name.lower()).count())

edge_counts = {}
for relation in namespace["RELATIONSHIPS"]:
    fact, dimension = relation["fromTable"], relation["toTable"]
    edge_name = f"edge_{fact}_{dimension}".lower()
    frame = spark.table(fact.lower()).select(
        F.concat(F.col("RecordID"), F.lit(":" + dimension)).alias("EdgeId"),
        F.col("RecordID").alias("SourceId"), F.col(relation["fromColumn"]).alias("TargetId")
    )
    if dimension == "DIM_DATE":
        frame = frame.withColumn("TargetDateInteger", F.date_format("TargetId", "yyyyMMdd").cast("long"))
    frame.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(edge_name)
    edge_counts[edge_name] = frame.count()

grain = ["ACCOUNT", "COSTCENTER", "PRODUCTLINE"]
actual = (spark.table("actuals").where((F.col("YEAR")==2026) & (F.col("PERIOD")<=9))
          .groupBy(*grain).agg(F.sum("Amount").alias("ActualAmount")))
plan = (spark.table("plan").where((F.col("YEAR")==2026) & (F.col("PERIOD")<=9))
        .groupBy(*grain).agg(F.sum("Amount").alias("PlanAmount")))
account_types = spark.table("account").select(F.col("Account").alias("ACCOUNT"), "AccountType", "FavorableSign")
drivers = (actual.join(plan, grain).join(account_types, "ACCOUNT")
    .withColumn("FavorableVariance", (F.col("ActualAmount")-F.col("PlanAmount"))*F.col("FavorableSign"))
    .withColumn("VariancePct", F.col("FavorableVariance")/F.col("PlanAmount"))
    .withColumn("ReviewRequired", (F.col("FavorableVariance")<=-100000) & (F.col("VariancePct")<=-0.10)))
drivers.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("variance_drivers")
print("Review flags support human review; no automated notifications or accounting actions.")


## Acceptance: persisted data, all 12 relationships and four SQL examples

In [ ]:
def scalar(value):
    if isinstance(value, (date, datetime)):
        return value.isoformat()
    if isinstance(value, Decimal):
        return str(value)
    return value

persisted = {
    name: [{key: scalar(value) for key, value in row.asDict().items()} for row in spark.table(name.lower()).collect()]
    for name in data
}
report = namespace["validate"](persisted)
if report["metrics"] != expected["metrics"] or report["tableRows"] != expected["tableRows"]:
    raise ValueError("Persisted Gold Delta tables differ from deterministic reference values")
report["edgeRows"] = edge_counts
report["reviewFlags"] = spark.table("variance_drivers").where("ReviewRequired").count()
fewshot_checks = []
for example in [{'question': 'Which cost centers and products have an unfavorable cloud expense variance YTD?', 'query': "SELECT a.COSTCENTER, c.Description AS CostCenterName, a.PRODUCTLINE,\nSUM(a.AmountCents) / 100.0 AS ActualUSD, SUM(p.AmountCents) / 100.0 AS PlanUSD,\n(SUM(p.AmountCents) - SUM(a.AmountCents)) / 100.0 AS FavorableVarianceUSD\nFROM actuals a JOIN plan p ON a.ACCOUNT=p.ACCOUNT AND a.COSTCENTER=p.COSTCENTER\nAND a.PRODUCTLINE=p.PRODUCTLINE AND a.DATE=p.DATE\nJOIN cost_center c ON a.COSTCENTER=c.CostCenter\nWHERE a.ACCOUNT='500100' AND a.YEAR=2026 AND a.PERIOD<=9\nGROUP BY a.COSTCENTER, c.Description, a.PRODUCTLINE\nHAVING SUM(a.AmountCents)>SUM(p.AmountCents)\nORDER BY FavorableVarianceUSD"}, {'question': 'Show the monthly evidence for Digital Banking cloud overspend.', 'query': "SELECT a.RecordID AS ActualRecordID, p.RecordID AS PlanRecordID, a.DATE,\na.Amount AS ActualUSD, p.Amount AS PlanUSD, p.Amount-a.Amount AS FavorableVarianceUSD\nFROM actuals a JOIN plan p ON a.ACCOUNT=p.ACCOUNT AND a.COSTCENTER=p.COSTCENTER\nAND a.PRODUCTLINE=p.PRODUCTLINE AND a.DATE=p.DATE\nWHERE a.ACCOUNT='500100' AND a.COSTCENTER='300110' AND a.PRODUCTLINE='DIGITAL'\nAND a.YEAR=2026 AND a.PERIOD<=9 ORDER BY a.DATE"}, {'question': 'Compare full-year revenue forecast and plan by product.', 'query': "WITH f AS (\nSELECT PRODUCTLINE, SUM(AmountCents)/100.0 AS ForecastUSD FROM forecast\nWHERE YEAR=2026 AND ACCOUNT IN ('400100','400200','400300') GROUP BY PRODUCTLINE\n), p AS (\nSELECT PRODUCTLINE, SUM(AmountCents)/100.0 AS PlanUSD FROM plan\nWHERE YEAR=2026 AND ACCOUNT IN ('400100','400200','400300') GROUP BY PRODUCTLINE\n)\nSELECT f.PRODUCTLINE, f.ForecastUSD, p.PlanUSD, f.ForecastUSD-p.PlanUSD AS VarianceUSD\nFROM f JOIN p ON f.PRODUCTLINE=p.PRODUCTLINE ORDER BY f.PRODUCTLINE"}, {'question': 'What is the actual revenue versus plan by month in 2026 through September?', 'query': "WITH a AS (\nSELECT PERIOD, SUM(AmountCents)/100.0 AS ActualUSD FROM actuals\nWHERE YEAR=2026 AND PERIOD<=9 AND ACCOUNT IN ('400100','400200','400300') GROUP BY PERIOD\n), p AS (\nSELECT PERIOD, SUM(AmountCents)/100.0 AS PlanUSD FROM plan\nWHERE YEAR=2026 AND PERIOD<=9 AND ACCOUNT IN ('400100','400200','400300') GROUP BY PERIOD\n)\nSELECT a.PERIOD, a.ActualUSD, p.PlanUSD, a.ActualUSD-p.PlanUSD AS VarianceUSD\nFROM a JOIN p ON a.PERIOD=p.PERIOD ORDER BY a.PERIOD"}]:
    result = spark.sql(example["query"])
    fewshot_checks.append({"question": example["question"], "rows": result.count()})
if [check["rows"] for check in fewshot_checks] != [36, 9, 6, 9]:
    raise ValueError(f"Unexpected few-shot results: {fewshot_checks}")
report["fewShots"] = fewshot_checks
notebookutils.fs.put(roots["gold"] + "/Files/fis_finance_iq/live-validation.json", json.dumps(report, indent=2), True)
print(json.dumps(report, indent=2))
display(spark.table("variance_drivers").where("ReviewRequired").orderBy("FavorableVariance"))
